# ml-boilerplate as a library — one task per cell

**Setup (pick one, then restart the kernel if you just installed):**
```bash
pip install "ml-boilerplate @ git+https://github.com/rajindersingh041/ml_boiler_plate.git"
# or: uv pip install "ml-boilerplate @ git+https://github.com/rajindersingh041/ml_boiler_plate.git"
# or: git pull this repo and work next to data/ (CSVs vendored, runs offline)
```
Each code cell does **exactly one task** and says why. Run top-to-bottom:
no YAML, no CLI, plain function calls. Data comes from `data/*.csv` when
present, otherwise from the built-in synthetic generators.

## 0. Imports — one line per capability so you can see what each module is for

In [ ]:
import tempfile  # stdlib: scratch dirs for the registry/EDA demo (nothing lands in the repo)
from pathlib import Path  # stdlib: locate the vendored data/ dir whatever the kernel cwd is

import numpy as np  # numeric helpers for the demo assertions/plots
import pandas as pd  # every builder takes DataFrames in and returns DataFrames/dicts out
from sklearn.pipeline import Pipeline  # we assemble preprocess + model ourselves, à la carte

from ml_boilerplate import registry  # save_run / promote / load_production (local model registry)
from ml_boilerplate.config import Config, ModelConfig  # Config() built in code replaces every YAML file
from ml_boilerplate.crossval import run_cv  # k-fold CV over the full pipeline
from ml_boilerplate.data import load_data  # synthetic fallback when no CSV is around
from ml_boilerplate.distributions import data_distributions, error_distributions  # JSON-able profiles
from ml_boilerplate.eda import profile_dataframe  # shape/dtypes/missing/stats dict
from ml_boilerplate.engineering import (  # datetime parts, holidays, lags, rolling windows
    add_datetime_features,
    add_lag_features,
)
from ml_boilerplate.evaluation import evaluate_on_test  # metrics + pass/fail release gate
from ml_boilerplate.io import read_table  # csv/parquet, local path or http(s) URL
from ml_boilerplate.model import build_model  # estimator by name from the task's registry
from ml_boilerplate.monitor import check_drift  # per-column PSI vs the training profile
from ml_boilerplate.tasks import get_task  # task object: splitter, preprocessor, metrics
from ml_boilerplate.validation import split_train_val_test  # one call, three splits

# Vendored CSVs live in <repo>/data no matter where the kernel was started.
# If this notebook ever runs without the repo around, DATA simply won't match
# and each loader cell falls back to synthetic data (see Cell 1).
DATA = next((p / 'data' for p in [Path.cwd(), *Path.cwd().parents]
               if (p / 'data' / 'mpg.csv').exists()), Path('data'))
print('imports ok — data dir:', DATA.resolve())

## 1. Load data — `read_table` handles a local path or URL; fall back to synthetic offline

In [ ]:
# Try the real Auto-MPG data first (398 rows, 9 cols, target = mpg).
# FileNotFoundError means 'running without the repo checkout' -> build the
# same-shaped problem synthetically instead, renaming its target to mpg so
# every later cell works unchanged.
try:
    df = read_table(str(DATA / 'mpg.csv'))
    print('vendored mpg.csv')
except FileNotFoundError:
    cfg0 = Config(task='regression')  # Config() with defaults, zero files involved
    df = load_data(cfg0.data, 'regression').rename(columns={'target': 'mpg'})
    print('synthetic fallback (target renamed to mpg)')
print(df.shape)
display(df.head(3))

## 2. Profile the frame — `profile_dataframe` returns plain dicts (no charts, no disk)

In [ ]:
# What's inside: n_rows/n_columns, per-column dtype, missing count+%, numeric
# summary stats, categorical cardinality. Say the numbers out loud in an
# interview: '398 rows, 6 missing horsepower, name is unique per row -> ID'.
prof = profile_dataframe(df)
print('rows/cols:', prof['n_rows'], prof['n_columns'])
print('missing:', {k: v for k, v in prof['missing'].items() if v['count']})
print('cardinality:', prof['categorical_cardinality'])

## 3. Data distributions — the JSON-able version of the same picture (feeds drift later)

In [ ]:
# data_distributions returns raw value lists (capped at 1000 per numeric col),
# the target's value counts, missing counts, cardinality. registry.save_run
# persists exactly this dict as profile.json, and check_drift (Cell 14)
# reuses its 'histograms' as the reference sample. Same object, three jobs.
dists = data_distributions(df, 'mpg')
print('keys:', sorted(dists.keys()))
print('mpg mean: %.1f | rows stored for histograms: %d' % (
    np.mean(dists['histograms']['mpg']), len(dists['histograms']['mpg'])))

## 4. Datetime features — row-local date parts on any date column (leak-free by construction)

In [ ]:
# Each output depends only on its own row's date, so this is safe to apply
# before any split. cyclical=True adds sin/cos pairs (month, day-of-week);
# holiday_country='US' adds is_holiday + days_to_holiday via the holidays lib.
try:
    sun = read_table(str(DATA / 'monthly-sunspots.csv'))
except FileNotFoundError:
    sun = pd.DataFrame({'Month': pd.date_range('2000-01-01', periods=24, freq='ME').astype(str),
                        'Sunspots': np.linspace(10, 50, 24)})
    print('synthetic sunspots fallback')
sun2 = add_datetime_features(sun, 'Month', cyclical=True, holiday_country='US')
print('new cols:', [c for c in sun2.columns if c.startswith('Month_')])
display(sun2.head(3))

## 5. Lag features — rolling stats built on `.shift(1)`, so the target never leaks into itself

In [ ]:
# The one rule that matters: every windowed stat sees only rows BEFORE the
# current one. Rows without enough history are dropped (the 'warm-up').
# lag_diffs adds target.diff(lag) — also past-only, hence also safe.
ts = pd.DataFrame({'date': pd.date_range('2024-01-01', periods=30, freq='D'),
                     'y': np.arange(30, dtype=float)})
lagged = add_lag_features(ts, target='y', date_col='date', n_lags=[1, 7],
                            rolling_windows=[7], rolling_stats=['mean', 'max'],
                            lag_diffs=True)
print('kept rows:', lagged.shape, '| warm-up rows dropped:', 30 - len(lagged))
display(lagged.head(3))

## 6. Split three ways — stratification/chronology chosen by task name, one call

In [ ]:
# Config() is just a dataclass: set whatever the YAML would have set.
# split_train_val_test returns (X_train, X_val, X_test, y_train, y_val, y_test).
# 'classification' stratifies; 'timeseries' slices chronologically (row order);
# anything else splits randomly. Same seed -> same splits, every run.
cfg = Config(task='regression')
cfg.data.test_size = 0.2
cfg.validation.val_size = 0.2
cfg.data.random_state = 42

X = df.drop(columns=['mpg'])
y = df['mpg']
Xtr, Xv, Xte, ytr, yv, yte = split_train_val_test(X, y, cfg, 'regression')
print(f'train={len(Xtr)}, val={len(Xv)}, test={len(Xte)}')

## 7. Build the pipeline — the task's shared preprocessor + a registry model, fitted once

In [ ]:
# get_task('regression') knows three things: how to split, which models are
# legal, and which metrics to report. build_preprocessor infers numeric vs
# categorical columns and imputes + scales + one-hots them. build_model
# resolves 'random_forest' against the REGRESSOR_REGISTRY. This exact
# Pipeline object is what CV tunes, the registry saves, and the API serves.
task = get_task('regression')
pipe = Pipeline([
    ('preprocess', task.build_preprocessor(Xtr, cfg)),
    ('model', build_model(ModelConfig(type='random_forest', params={'n_estimators': 50}),
                           task.model_registry())),
])
pipe.fit(Xtr, ytr)
pred = pipe.predict(Xte)
print('R2 on test: %.3f' % pipe.score(Xte, yte))

## 8. Cross-validate — same pipeline refit per fold; aggregate is mean ± std

In [ ]:
# run_cv rebuilds the preprocess+model pipeline inside each fold (no leakage
# across folds) and scores with the task's own metrics, so these numbers
# match train.py exactly. Returns {'per_fold': [...], 'aggregate': {...}}.
# StratifiedKFold for classification, TimeSeriesSplit for timeseries.
cfg.cross_validation.folds = 3
cv = run_cv(Xtr, ytr, cfg, 'regression')
print('CV R2: %.3f ± %.3f' % (cv['aggregate']['r2']['mean'], cv['aggregate']['r2']['std']))

## 9. Release gate — metrics vs minimum thresholds, pass/fail as data

In [ ]:
# evaluate_on_test is pure: arrays in, {'metrics', 'passed', 'failed'} out.
# The CLI turns passed=False into exit code 1, so this doubles as a CI gate.
# min_metrics keys must exist in the task's metric set (else they count as failed).
gate = evaluate_on_test(yte.to_numpy(), pred, None, 'regression', {'r2': 0.5})
print('gate passed:', gate['passed'], '| failed:', gate['failed'])
print('metrics:', {k: round(v, 4) for k, v in gate['metrics'].items()})

## 10. Error distributions — residuals, the shape the metrics hide

In [ ]:
# Means lie; residuals don't. For regression you get the raw residual list +
# RMSE (plot it: skew | funnel = bias | heteroscedasticity). For
# classification you get the confusion matrix (+ ROC AUC when binary
# probabilities are passed). Same dict the CLI turns into HTML charts.
err = error_distributions(yte.to_numpy(), pred, None, 'regression')
print('keys:', sorted(err.keys()))
print('first residuals:', np.round(err['residuals'][:5], 3))

## 11. Full EDA bundle — report JSON + three plotly charts, one call

In [ ]:
# run_eda = profile_dataframe + missing-values bar + histograms +
# correlation heatmap, written to output_dir. Here we point it at a scratch
# dir so the repo stays clean; the CLI points it at artifacts/<task>/plots.
from ml_boilerplate.config import EdaConfig
from ml_boilerplate.eda import run_eda
eda_out = run_eda(df, EdaConfig(enabled=True, output_dir=str(tmp_eda := tempfile.mkdtemp())),
                report_path=str(Path(tmp_eda) / 'eda_report.json'))
print(sorted(eda_out.keys()))

## 12. Tune — grid search over `model__*` params on the train split only

In [ ]:
# Param keys are prefixed with the pipeline step name ('model__'), so the
# search can only move model knobs — never the preprocessor. method='grid'
# tries every combo; 'random' samples n_iter of them. Tune on train, judge
# on the untouched test split (next line prints both: no peeking).
from ml_boilerplate.config import TuningConfig
from ml_boilerplate.tuning import run_tuning
cfg.tuning = TuningConfig(enabled=True, method='grid',
                            param_grid={'model__n_estimators': [10, 25],
                                        'model__max_depth': [3, None]})
best_pipe, tune_res = run_tuning(Xtr[:150], ytr[:150], cfg, 'regression')
print('best:', tune_res['best_params'])
print('tuned R2 on held-out test: %.3f' % best_pipe.score(Xte, yte))

## 13. Register — version the pipeline + metrics + training profile, then promote

In [ ]:
# save_run writes runs/<utc-id>/{model.joblib, metrics.json, params.json,
# profile.json, config.json}. promote() copies a run to production/ atomically
# (tmp-dir + rename: readers never see a half-copied model). load_production
# raises FileNotFoundError when nothing was ever promoted — the CLI and the
# API both rely on that contract to fall back / fail loudly.
tmp = Path(tempfile.mkdtemp())
run_id = registry.save_run(str(tmp / 'runs'), pipe,
                             metrics={'r2': gate['metrics']['r2']},
                             params={'n_estimators': 50},
                             profile=data_distributions(df.assign(mpg=y), 'mpg'),
                             cfg_snapshot={'task': 'regression'})
registry.promote(str(tmp / 'runs'), str(tmp / 'prod'), run_id)
loaded = registry.load_production(str(tmp / 'prod'))
print('promoted', run_id, '| reloaded model predicts:', len(loaded.predict(Xte)), 'rows')

## 14. Predict from a file — the same driver the CLI calls

In [ ]:
# run_prediction(cfg, input, output): load model (production first,
# artifacts.model_path fallback) -> task.prepare_predict_features (timeseries
# rebuilds lags from history here) -> write predictions (+probability for
# binary). We persist the fitted pipe and point the config at it first.
import joblib
from ml_boilerplate.io import write_table
from ml_boilerplate.predict import run_prediction
model_path = str(tmp / 'model.joblib')
joblib.dump(pipe, model_path)
cfg.artifacts.model_path = model_path
write_table(Xte[:5], str(tmp / 'new.csv'))
out_path = run_prediction(cfg, str(tmp / 'new.csv'), str(tmp / 'preds.csv'))
display(read_table(out_path).head())

## 15. Serve it — the promoted model behind GET /health + POST /predict

In [ ]:
# create_app(production_dir, task, target) is the FastAPI factory the
# Dockerfile serves (uvicorn ml_boilerplate.api:app). /predict takes
# {'rows': [{col: val, ...}]} and runs the same prepare_predict_features as
# training, so serving can't skew from train. Numpy scalars aren't JSON
# serializable, hence the .item() conversion below (pandas -> stdlib).
try:
    from fastapi.testclient import TestClient  # needs httpx (dev dep)
    from ml_boilerplate.api import create_app
except ImportError as e:
    print('serve demo skipped (pip install fastapi httpx):', e)
else:
    app = create_app(str(tmp / 'prod'), 'regression', 'mpg')
    client = TestClient(app)
    print('health:', client.get('/health').json())
    rows = [{k: (v.item() if isinstance(v, np.generic) else v)
             for k, v in r.items()} for r in Xte[:2].to_dict(orient='records')]
    print('POST /predict ->', client.post('/predict', json={'rows': rows}).json())

## 16. Monitor drift — same batch vs shifted batch, PSI tells them apart

In [ ]:
# check_drift compares each numeric column's live values against the stored
# histogram reference (Cell 3's dict — the third job of that object).
# PSI > 0.25 per column = drifted; the CLI exits 1 on that, so this is the
# same CI gate, called directly. Identical data scores ~0 by construction.
profile = data_distributions(df, 'mpg')
print('identical batch drifted:', check_drift(profile, df, psi_threshold=0.25)['drifted'])
shifted = df.copy(); shifted['weight'] = shifted['weight'] * 3
print('weight x3 drifted:', check_drift(profile, shifted, psi_threshold=0.25)['drifted'])

## Cheat sheet — what to import for what

| Need | Import |
|---|---| 
| Read csv/parquet, local or URL | `from ml_boilerplate.io import read_table, write_table` |
| Synthetic data (offline/tests) | `from ml_boilerplate.data import load_data` |
| EDA profile dict | `from ml_boilerplate.eda import profile_dataframe, run_eda` |
| Datetime / lag / rolling features | `from ml_boilerplate.engineering import add_datetime_features, add_lag_features, featurize` |
| Train/val/test split | `from ml_boilerplate.validation import split_train_val_test` |
| Shared preprocessor | `task.build_preprocessor(X, cfg)` via `get_task(name)` |
| Model by name | `from ml_boilerplate.model import build_model` + registries |
| K-fold CV | `from ml_boilerplate.crossval import run_cv` |
| Grid/random tuning | `from ml_boilerplate.tuning import run_tuning` |
| Full EDA bundle (report + charts) | `from ml_boilerplate.eda import run_eda` |
| Test gate | `from ml_boilerplate.evaluation import evaluate_on_test` |
| Error shapes | `from ml_boilerplate.distributions import error_distributions` |
| Save/promote/load model | `from ml_boilerplate import registry` |
| Score a file (predict driver) | `from ml_boilerplate.predict import run_prediction` |
| Drift check | `from ml_boilerplate.monitor import check_drift` |
| Serve over HTTP | `from ml_boilerplate.api import create_app` |

`Config(task=...)` built in code replaces every YAML file; only `run_training`/`main` need YAML.